# Surya Embedding Space Analysis

This notebook extracts backbone embeddings from the pretrained Surya model,
visualizes them in 2D using UMAP, and quantifies label separation with a
silhouette score.

**Workflow:**
1. Load the SDO data index and find samples closest to target date(s)
2. Subsample the index for a tractable embedding run
3. Initialize the Surya backbone and load pretrained weights
4. Extract per-sample embeddings (global average pool over patch tokens)
5. Align labels from an external file to the embedded samples
6. Project embeddings to 2D with UMAP and color by label
7. Compute the silhouette score

**Extending this notebook:**
- Aggregate embeddings from multiple index files: see *Section 2 – Subsample*
- Continuous labels (e.g. X-ray flux): swap `label_type = 'continuous'` in *Section 5*

## Imports

In [ ]:
import sys
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tqdm.auto import tqdm

# ── project root on sys.path ───────────────────────────────────────────────
# Locate the repo root by walking up until we find the Surya submodule.
# Adjust if you launch this notebook from a different working directory.
_notebook_dir = Path.cwd()
_repo_root = _notebook_dir
while not (_repo_root / "Surya").exists() and _repo_root != _repo_root.parent:
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from workshop_infrastructure.datasets.helio import HelioNetCDFDataset
from workshop_infrastructure.models.helio_spectformer import HelioSpectFormer
from workshop_infrastructure.utils import load_pretrained_weights, build_scalers

## Configuration

All tunable parameters live here. Edit this cell before running the rest.

In [ ]:
# ── Data paths ─────────────────────────────────────────────────────────────
INDEX_PATH = "/nobackupnfs1/sroy14/processed_data/Helio/csv_files/full_data_201006_to_202412_with_priority.csv"
SCALERS_PATH = str(_repo_root / "downstream_apps/template/assets/scalers.yaml")
CHECKPOINT_PATH = str(_repo_root / "downstream_apps/template/assets/surya.366m.v1.pt")

# ── Label file ─────────────────────────────────────────────────────────────
# Provide a CSV with at least one datetime column and one label column.
# Set to None to derive labels directly from the index CSV (xrsb_flux → GOES class).
LABEL_FILE_PATH = str(_repo_root / "downstream_apps/template/data/hek_flare_catalog.csv")
LABEL_TIME_COL  = "start_time"   # column in the label file used for temporal alignment
LABEL_VALUE_COL = "GOES_class"   # column carrying the label value
LABEL_TIME_TOLERANCE = pd.Timedelta("6h")  # max gap between a sample and its matched label

# ── Analysis parameters ────────────────────────────────────────────────────
TARGET_DATE  = "2014-06-01"  # reference datestamp for single-sample demo (Section 1)
START_DATE   = "2014-01-01"  # start of date window for batch embedding
END_DATE     = "2014-12-31"  # end   of date window for batch embedding
MAX_SAMPLES  = 200           # cap on the number of samples to embed
RANDOM_SEED  = 42

# ── Model architecture (must match the pretrained checkpoint) ───────────────
CHANNELS = [
    "aia94", "aia131", "aia171", "aia193", "aia211",
    "aia304", "aia335", "aia1600",
    "hmi_m", "hmi_bx", "hmi_by", "hmi_bz", "hmi_v",
]
MODEL_CONFIG = dict(
    img_size          = 4096,
    patch_size        = 16,
    in_chans          = 13,
    embed_dim         = 1280,
    time_embedding    = {"type": "linear", "time_dim": 1},
    depth             = 10,
    n_spectral_blocks = 2,
    num_heads         = 16,
    mlp_ratio         = 4.0,
    drop_rate         = 0.0,
    window_size       = 2,
    dp_rank           = 4,
    nglo              = 1,
    checkpoint_layers = list(range(10)),
    finetune          = True,   # return patch tokens instead of reconstructed image
)

# ── UMAP parameters ────────────────────────────────────────────────────────
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST    = 0.1

# ── Output directory ───────────────────────────────────────────────────────
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

---
## Section 1 — Load the CSV Index & Find the Closest Sample

Demonstrates how to locate the SDO stack whose timestamp is closest to an
arbitrary target date.

In [ ]:
index_df = pd.read_csv(INDEX_PATH)
index_df["timestep"] = pd.to_datetime(index_df["timestep"])
index_df = index_df[index_df["present"] == 1].reset_index(drop=True)

print(f"Index loaded: {len(index_df):,} valid samples  "
      f"({index_df['timestep'].min().date()} → {index_df['timestep'].max().date()})")
print("Columns:", list(index_df.columns))
index_df.head(3)

In [ ]:
def find_closest_sample(df: pd.DataFrame, target_date: str) -> pd.Series:
    """Return the row in *df* whose timestep is closest to *target_date*."""
    target = pd.Timestamp(target_date)
    idx = (df["timestep"] - target).abs().idxmin()
    return df.loc[idx]


closest = find_closest_sample(index_df, TARGET_DATE)
print(f"Target  : {TARGET_DATE}")
print(f"Closest : {closest['timestep']}  (Δt = {abs(closest['timestep'] - pd.Timestamp(TARGET_DATE))})")
print(f"Path    : {closest['path']}")

---
## Section 2 — Subsample the Index for Batch Embedding

Filters the index to a date window and draws up to `MAX_SAMPLES` rows
uniformly at random. The result is saved to `outputs/embedding_index.csv`
for reproducibility.

**To aggregate embeddings from multiple index files**, load each CSV,
filter and subsample as below, then `pd.concat` before saving.

In [ ]:
mask = (
    (index_df["timestep"] >= pd.Timestamp(START_DATE))
    & (index_df["timestep"] <= pd.Timestamp(END_DATE))
)
window_df = index_df[mask].reset_index(drop=True)
print(f"Samples in [{START_DATE}, {END_DATE}]: {len(window_df):,}")

# ── uniform temporal subsampling ───────────────────────────────────────────
# Draws samples at roughly equal intervals across the window, giving better
# coverage than a purely random draw when the window spans many months.
if len(window_df) > MAX_SAMPLES:
    step = len(window_df) // MAX_SAMPLES
    embed_df = window_df.iloc[::step].head(MAX_SAMPLES).reset_index(drop=True)
else:
    embed_df = window_df.copy()

# To aggregate from multiple index files, concat additional filtered DataFrames here:
#   extra_df = pd.read_csv(EXTRA_INDEX_PATH)
#   extra_df["timestep"] = pd.to_datetime(extra_df["timestep"])
#   embed_df = pd.concat([embed_df, extra_df_filtered], ignore_index=True)

embed_index_path = OUTPUT_DIR / "embedding_index.csv"
embed_df.to_csv(embed_index_path, index=False)
print(f"Embedding index: {len(embed_df)} samples  →  {embed_index_path}")

---
## Section 3 — Initialize the Surya Backbone

Instantiates the 366M-parameter HelioSpectFormer with `finetune=True`
(strips the pretraining decoder so the backbone returns patch tokens)
and loads the pretrained checkpoint.

In [ ]:
backbone = HelioSpectFormer(**MODEL_CONFIG)

# load_pretrained_weights handles flat checkpoint keys (saved from HelioSpectFormer)
# and nested keys (saved from HelioSpectformer1D) transparently.
load_pretrained_weights(backbone, CHECKPOINT_PATH)

backbone.eval()
backbone.to(DEVICE)

n_params = sum(p.numel() for p in backbone.parameters())
print(f"Backbone parameters: {n_params / 1e6:.1f} M")

---
## Section 4 — Extract Embeddings

Builds a `HelioNetCDFDataset` from the filtered index and runs each
sample through the backbone.

**Backbone output** (with `finetune=True`, `nglo=1`):
- shape `(B, N_patches + 1, embed_dim)` where N_patches = (4096 / 16)² = 65 536
- Token `[:, 0, :]` is a learned global token; the rest are patch tokens.

We use **global average pooling** over *all* tokens because it does not
depend on a learned parameter and is stable for zero-shot analysis.

In [ ]:
scalers = build_scalers(SCALERS_PATH)

dataset = HelioNetCDFDataset(
    index_path              = str(embed_index_path),
    time_delta_input_minutes= [0],   # single frame at reference time
    time_delta_target_minutes= 60,   # unused; load_forecast_frames=False
    n_input_timestamps      = 1,
    rollout_steps           = 0,
    scalers                 = scalers,
    channels                = CHANNELS,
    phase                   = "embed",
    load_forecast_frames    = False,  # skip future-frame fetching
)

# shuffle=False keeps valid_indices order aligned with extracted embeddings
loader = torch.utils.data.DataLoader(
    dataset,
    batch_size  = 1,
    shuffle     = False,
    num_workers = 2,
    pin_memory  = DEVICE.type == "cuda",
)

print(f"Dataset size: {len(dataset)} samples")
print(f"Timestamps  : {dataset.valid_indices[0]} → {dataset.valid_indices[-1]}")

In [ ]:
@torch.no_grad()
def extract_embeddings(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    device: torch.device,
) -> np.ndarray:
    """
    Run all samples through *model* and return a (N, embed_dim) embedding matrix.

    Each sample's embedding is the global average of its patch tokens,
    cast to float32 regardless of the model's internal dtype.
    """
    model.eval()
    all_embeddings = []

    for batch in tqdm(dataloader, desc="Extracting embeddings"):
        batch = {
            k: v.to(device) if isinstance(v, torch.Tensor) else v
            for k, v in batch.items()
        }
        tokens = model(batch)                  # (B, N_tokens, embed_dim)
        embeddings = tokens.mean(dim=1)        # (B, embed_dim) — global average pool
        all_embeddings.append(embeddings.cpu().float().numpy())

    return np.vstack(all_embeddings)           # (N_samples, embed_dim)


embeddings = extract_embeddings(backbone, loader, DEVICE)

# timestamps in the same order as embeddings (shuffle=False guarantees this)
timestamps = pd.to_datetime([str(ts) for ts in dataset.valid_indices])

print(f"Embeddings shape: {embeddings.shape}")
np.save(OUTPUT_DIR / "embeddings.npy", embeddings)
np.save(OUTPUT_DIR / "timestamps.npy", timestamps.values)

---
## Section 5 — Load & Align Labels

Labels are loaded from an external CSV and aligned to the embedded samples
via `pd.merge_asof` (nearest-timestamp join with a configurable tolerance).

**Categorical labels** (default): GOES class letter extracted from `GOES_class`
(e.g. `"M1.2"` → `"M"`).

**Continuous labels**: set `label_type = 'continuous'` and point
`LABEL_VALUE_COL` at a numeric column (e.g. `"intensity"` in the flare
catalog, or `"xrsb_flux"` in the index CSV).

In [ ]:
label_type = "categorical"   # "categorical" | "continuous"

# ── Load label file ────────────────────────────────────────────────────────
label_df = pd.read_csv(LABEL_FILE_PATH)
label_df[LABEL_TIME_COL] = pd.to_datetime(label_df[LABEL_TIME_COL], utc=False)
label_df = label_df.sort_values(LABEL_TIME_COL).reset_index(drop=True)

# ── Build sample DataFrame ─────────────────────────────────────────────────
samples_df = pd.DataFrame({"timestep": timestamps})
samples_df = samples_df.sort_values("timestep").reset_index(drop=True)

# ── Nearest-timestamp join ─────────────────────────────────────────────────
# merge_asof requires both keys to be sorted.
joined = pd.merge_asof(
    samples_df,
    label_df[[LABEL_TIME_COL, LABEL_VALUE_COL]].rename(columns={LABEL_TIME_COL: "timestep"}),
    on           = "timestep",
    direction    = "nearest",
    tolerance    = LABEL_TIME_TOLERANCE,
)

n_matched = joined[LABEL_VALUE_COL].notna().sum()
print(f"Samples with matched label: {n_matched} / {len(joined)}")

# keep only matched rows and reorder embeddings to match
# (samples_df was sorted by timestep, so we must reorder embeddings accordingly)
original_order = pd.DataFrame({"timestep": timestamps}).reset_index()
joined = joined.merge(original_order, on="timestep", how="left")
matched_mask_sorted = joined[LABEL_VALUE_COL].notna()
matched_indices = joined.loc[matched_mask_sorted, "index"].values

matched_embeddings = embeddings[matched_indices]
raw_labels         = joined.loc[matched_mask_sorted, LABEL_VALUE_COL].values

# ── Parse label values ─────────────────────────────────────────────────────
if label_type == "categorical":
    # Extract leading letter from GOES class strings like "M1.2" → "M"
    labels = np.array([str(v)[0].upper() for v in raw_labels])
    label_order = [c for c in ["A", "B", "C", "M", "X"] if c in np.unique(labels)]
    print("Label distribution:")
    for cls in label_order:
        print(f"  {cls}: {(labels == cls).sum()}")

elif label_type == "continuous":
    labels = raw_labels.astype(float)
    print(f"Label range: [{labels.min():.3e}, {labels.max():.3e}]")

print(f"\nEmbeddings for visualization: {matched_embeddings.shape}")

---
## Section 6 — 2D UMAP Visualization

In [ ]:
try:
    import umap
except ImportError:
    raise ImportError("umap-learn is required: pip install umap-learn")

reducer = umap.UMAP(
    n_neighbors  = UMAP_N_NEIGHBORS,
    min_dist     = UMAP_MIN_DIST,
    n_components = 2,
    random_state = RANDOM_SEED,
    verbose      = True,
)
embedding_2d = reducer.fit_transform(matched_embeddings)
print(f"2D projection shape: {embedding_2d.shape}")

np.save(OUTPUT_DIR / "embedding_2d.npy", embedding_2d)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

if label_type == "categorical":
    # assign a fixed color to each GOES class
    palette = {"A": "#aec6cf", "B": "#90ee90", "C": "#ffd700",
               "M": "#ff8c00", "X": "#dc143c"}
    for cls in label_order:
        mask = labels == cls
        ax.scatter(
            embedding_2d[mask, 0], embedding_2d[mask, 1],
            c      = palette.get(cls, "grey"),
            label  = f"GOES {cls} (n={mask.sum()})",
            s      = 20,
            alpha  = 0.7,
            edgecolors = "none",
        )
    ax.legend(title="Flare class", loc="best", markerscale=2)

elif label_type == "continuous":
    sc = ax.scatter(
        embedding_2d[:, 0], embedding_2d[:, 1],
        c     = np.log10(np.abs(labels) + 1e-12),
        cmap  = "plasma",
        s     = 20,
        alpha = 0.7,
        edgecolors = "none",
    )
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(f"log₁₀({LABEL_VALUE_COL})")

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title(
    f"Surya embedding space — UMAP\n"
    f"{START_DATE} → {END_DATE}  (n={len(matched_embeddings)})"
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "umap.png", dpi=150)
plt.show()

---
## Section 7 — Silhouette Score

The **silhouette score** (range −1 to +1) measures how tightly samples
cluster within their label group relative to neighboring groups.
Values near +1 indicate well-separated clusters; near 0, overlapping;
near −1, systematic misassignment.

For **continuous labels**, we report instead the Spearman rank correlation
between pairwise embedding distances and pairwise label differences,
which measures how well the geometry of the embedding space tracks the
label ordering.

In [ ]:
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.preprocessing import LabelEncoder

if label_type == "categorical":
    if len(np.unique(labels)) < 2:
        print("Need at least 2 distinct classes to compute silhouette score.")
    else:
        le = LabelEncoder()
        labels_int = le.fit_transform(labels)

        # overall score (on high-dim embeddings — more informative than on 2D)
        score = silhouette_score(matched_embeddings, labels_int, metric="cosine")
        print(f"Silhouette score (cosine, {len(np.unique(labels))} classes): {score:.4f}")

        # per-sample scores for a breakdown by class
        sample_scores = silhouette_samples(matched_embeddings, labels_int, metric="cosine")

        print("\nPer-class mean silhouette:")
        for cls_name, cls_idx in zip(le.classes_, range(len(le.classes_))):
            cls_mask = labels_int == cls_idx
            if cls_mask.sum() > 0:
                print(f"  {cls_name}: {sample_scores[cls_mask].mean():.4f}  (n={cls_mask.sum()})")

        # per-class silhouette bar chart
        fig, ax = plt.subplots(figsize=(6, 3))
        cls_means = [
            sample_scores[labels_int == i].mean()
            for i in range(len(le.classes_))
        ]
        ax.bar(le.classes_, cls_means, color="steelblue")
        ax.axhline(score, color="red", linestyle="--", label=f"Overall: {score:.3f}")
        ax.set_xlabel("GOES class")
        ax.set_ylabel("Mean silhouette")
        ax.set_title("Silhouette score by GOES class")
        ax.legend()
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "silhouette.png", dpi=150)
        plt.show()

elif label_type == "continuous":
    from scipy.stats import spearmanr
    from sklearn.metrics import pairwise_distances

    # subsample pairwise distances to avoid O(N²) cost for large N
    N = len(matched_embeddings)
    rng = np.random.default_rng(RANDOM_SEED)
    idx = rng.choice(N, size=min(N, 500), replace=False)

    emb_sub    = matched_embeddings[idx]
    label_sub  = labels[idx]

    emb_dists   = pairwise_distances(emb_sub, metric="cosine").ravel()
    label_dists = np.abs(label_sub[:, None] - label_sub[None, :]).ravel()

    rho, pval = spearmanr(emb_dists, label_dists)
    print(f"Spearman ρ (embedding dist vs. |label diff|): {rho:.4f}  (p={pval:.2e})")
    print("Positive ρ → samples with similar labels are closer in embedding space.")